In [1]:
# Tecnología
import json
import calendar
import pandas as pd
from sparky_bc import Sparky
import datetime as dt
from dateutil.relativedelta import relativedelta

# files lz conection
path_sparky_conf = '/Users/santlond/Documents/sparky_conf.json'

# Configurar conexión a LZ
with open(path_sparky_conf, 'rb') as JSON_lz_File:
    sp_config = json.loads(JSON_lz_File.read())
    
USER='santlond'
PASS=sp_config['ID']
DSN='IMPALA_PROD'
LOGDIR= 'logs'
# sparky = Sparky(username=USER, password=PASS, dsn=DSN, hostname="sbmdeblze004.bancolombia.corp")
sparky = Sparky(username=USER, password=PASS, dsn=DSN, hostname="sbmdeblze004.bancolombia.corp", spark_submit="spark3-submit")
 
# sparky = Sparky(username=USER, password=PASS, dsn=DSN)

helper = sparky.helper

/Users/santlond/Documents/venv_py39_odbc/lib/python3.9/site-packages/helper/helper.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2026-03-25 16:01:55 - [WARNING] - No se encontro la carpeta "/Users/santlond/Documents/ADQUIRENCIA_FERIA_EVA/logs" para guardar los logs


 ____  _____ __  __  ___ _____ _____ 
|  _ \| ____|  \/  |/ _ \_   _| ____|
| |_) |  _| | |\/| | | | || | |  _|  
|  _ <| |___| |  | | |_| || | | |___ 
|_| \_\_____|_|  |_|\___/ |_| |_____|
                                     
 ____  ____   _    ____  _  __
/ ___||  _ \ / \  |  _ \| |/ /
\___ \| |_) / _ \ | |_) | ' / 
 ___) |  __/ ___ \|  _ <| . \ 
|____/|_| /_/   \_\_| \_\_|\_\
                              



# Tabla resultados_wompi.wompi_businesses_procedures

In [2]:
dict_ult_ing_wompi_busines = helper.obtener_ultima_ingestion('resultados_wompi.wompi_businesses_procedures')
dict_ult_ing_wompi_busines

2026-03-25 16:02:01 - [INFO] - Buscando fechas para resultados_wompi.wompi_businesses_procedures
2026-03-25 16:02:01 - [INFO] - Transcurrido: 1774472521, Tiempo de Refresco = 1000
/Users/santlond/Documents/venv_py39_odbc/lib/python3.9/site-packages/helper/helper.py:476: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, cn)
2026-03-25 16:02:04 - [INFO] - Finalizo la busqueda, duracion: 00:03.0, resultado: {'year': 2026, 'month': 3, 'day': 25}


{'year': 2026, 'month': 3, 'day': 25}

In [ ]:
sql = """
select *
from resultados_wompi.wompi_businesses_procedures
where year = """ + str(dict_ult_ing_wompi_busines['year']) + """
and month = """ + str(dict_ult_ing_wompi_busines['month']) + """
and day = """ + str(dict_ult_ing_wompi_busines['day']) + """
and id_comercio = 100021
-- and id_comercio = 196714
order by fecha_actualizacion_procedimiento desc;
"""
print(sql)


select *
from resultados_wompi.wompi_businesses_procedures
where year = 2026
and month = 3
and day = 25
and id_comercio = 100021
order by fecha_actualizacion_procedimiento desc;



# Tabla resultados_wompi.wompi_merchants

## Hallazgos


Tabla `resultados_wompi.wompi_merchants`

- La tabla se ingesta full, pero se borran las ingestas full del pasado. Por tanto no permitiría generar el histórico de vinculados activos en cada mes.
- Los registros son únicos por la variable id_comercio
- Un comercio puede tener varios id_comercio ¿Cómo interpretar esto? ¿Tiene sentido? ¿Cómo gestionarlo? ya que por documento_identidad y tipo_documento hay varios registros. Ejemplo: 800253799. Respuesta: Un comercio como Frisby puede tener varios locales y cada uno un id_comercio [id wompi] diferente
- Modelo = 'Agregador' porque significan que dan datáfono, link pagos por pse y botón bancolombia. 'Gateway' acopla todos los medios de pago vinculados al pago se acopla la adquirencia entonces se estaría doble contando. Otra explicación que da *Daniel Ramirez Vergara* es Wompi Agregador es una solucion que entrega los medios de pago (adquirencia, botones, puntos y demas) Gateway es una forma de llevar los productos que el cliente tiene con bancolombia al mundo digital ! 


### Datos importantes

- Desde Junio 2025 [aunque se ve un aumento significativo en mayo 2025] se iniciaron las acciones comerciales desde el equipo comercial. [Tanto adquirencia como wompi]
- En agosto inicio nequi negocios


¿Qué tal este query para obtener el historico de vinculaciones a wompi?

In [6]:
dict_ult_ing_wompi_merch = helper.obtener_ultima_ingestion('resultados_wompi.wompi_merchants')
dict_ult_ing_wompi_merch

2026-03-25 16:17:15 - [INFO] - Buscando fechas para resultados_wompi.wompi_merchants
/Users/santlond/Documents/venv_py39_odbc/lib/python3.9/site-packages/helper/helper.py:476: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, cn)
2026-03-25 16:17:16 - [INFO] - Finalizo la busqueda, duracion: 00:01.4, resultado: {'year': 2026, 'month': 3, 'day': 25}


{'year': 2026, 'month': 3, 'day': 25}

In [ ]:
sql = """
SELECT CASt(left(cast(creado as string), 6) as int) as creado_ym, count(*) as num_vinc
FROM resultados_wompi.wompi_merchants
WHERE YEAR = """ + str(dict_ult_ing_wompi_merch['year']) + """
  AND MONTH = """ + str(dict_ult_ing_wompi_merch['month']) + """
  AND DAY = """ + str(dict_ult_ing_wompi_merch['day']) + """
  AND modelo = 'Agregador'
  and activo = 'A'
  and desembolsos_permitidos = 'Si'
GROUP BY 1
ORDER BY creado_ym;
"""
helper.obtener_dataframe(sql)



------------------------------------------------------------
  i    tipo    nombre    estado     hora_inicio   duracion   
------------------------------------------------------------
 4/4 DATAFRAME         ejecutando   08:54:16 AM             

2025-11-13 08:54:17 - [INFO] - 86 filas, 2 columnas, 00:00.6 consultando, 00:00.0 descargando, 00:00.0 convirtiendo


 4/4 DATAFRAME         finalizado   08:54:16 AM     00:00.7 
------------------------------------------------------------


,creado_ym,num_vinc
0,201807,1
1,201810,1
2,201811,1
3,201812,1
4,201902,1
...,...,...
81,202507,4330
82,202508,7090
83,202509,14925
84,202510,12637


In [12]:
# Vinculación mensual

sql_drop = """DROP TABLE IF EXISTS proceso.mdo_aceptacion_comercios_hist_vinc_wompi PURGE;"""
helper.ejecutar_consulta(sql_drop)

sql = """
CREATE TABLE proceso.mdo_aceptacion_comercios_hist_vinc_wompi STORED AS PARQUET AS
WITH conteo AS (
SELECT CASt(left(cast(creado as string), 6) as int) as fecha_ym, 
       count(*) as num_vinc_new
FROM resultados_wompi.wompi_merchants
WHERE YEAR = """ + str(dict_ult_ing_wompi_merch['year']) + """
  AND MONTH = """ + str(dict_ult_ing_wompi_merch['month']) + """
  AND DAY = """ + str(dict_ult_ing_wompi_merch['day']) + """
        AND modelo = 'Agregador'
        AND activo = 'A'
        AND desembolsos_permitidos = 'Si'
        GROUP BY 1),
        conteo_y_m AS
        (SELECT fecha_ym,
                num_vinc_new,
                cast(left(cast(fecha_ym AS STRING), 4) AS int) AS YEAR,
                cast(right(cast(fecha_ym AS STRING), 2) AS int) AS mes,
                1 AS paracumsum
        FROM conteo)
        SELECT fecha_ym,
        num_vinc_new,
        sum(num_vinc_new) OVER (PARTITION BY YEAR
                                ORDER BY YEAR, mes ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS num_vinc_new_cumsum_ym,
        sum(num_vinc_new) OVER (PARTITION BY paracumsum
                                ORDER BY fecha_ym) AS num_vinc_new_cumsum,
        'wompi' AS producto
        FROM conteo_y_m
        ORDER BY fecha_ym DESC;
"""
helper.ejecutar_consulta(sql)

sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_aceptacion_comercios_hist_vinc_wompi;"""
helper.ejecutar_consulta(sql_compute)

------------------------------------------------------------------------------------------
  i  tipo                  nombre                     estado     hora_inicio   duracion   
------------------------------------------------------------------------------------------
 1/1 DROP ..._aceptacion_comercios_hist_vinc_wompi   finalizado   11:38:08 AM     00:00.2 
------------------------------------------------------------------------------------------
--------------------------------------------------------------------------------------------
  i   tipo                   nombre                     estado     hora_inicio   duracion   
--------------------------------------------------------------------------------------------
 2/2 CREATE ..._aceptacion_comercios_hist_vinc_wompi   finalizado   11:38:08 AM     00:47.4 
--------------------------------------------------------------------------------------------
--------------------------------------------------------------------------------

In [14]:
sql = """
select 
fecha_ym,
        num_vinc_new,
        num_vinc_new_cumsum_ym,
        num_vinc_new_cumsum
from proceso.mdo_aceptacion_comercios_hist_vinc_wompi
order by fecha_ym desc;
"""
helper.obtener_dataframe(sql).head(40)

-----------------------------------------------------------------------------------------------
  i    tipo                     nombre                     estado     hora_inicio   duracion   
-----------------------------------------------------------------------------------------------
 5/5 DATAFRAME                                            ejecutando   11:44:02 AM             

2026-03-24 11:45:30 - [INFO] - 90 filas, 4 columnas, 01:27.3 consultando, 00:00.0 descargando, 00:00.0 convirtiendo


 5/5 DATAFRAME                                            finalizado   11:44:02 AM     01:27.3 
-----------------------------------------------------------------------------------------------


,fecha_ym,num_vinc_new,num_vinc_new_cumsum_ym,num_vinc_new_cumsum
0,202603,13745,38764,184790
1,202602,15144,25019,171045
2,202601,9875,9875,155901
3,202512,9543,74328,146026
4,202511,11918,64785,136483
...,...,...,...,...
85,201902,1,1,5
86,201812,1,4,4
87,201811,1,3,3
88,201810,1,2,2
